# Data Understanding & Wrangling - Ames Housing

This notebook focuses on understanding, validating, and preparing the Ames Housing dataset for subsequent analysis and predictive modeling. The objective is to clean the data, address missing values and inconsistencies, and produce a reliable dataset for predicting **`SalePrice`** based on property characteristics such as size, location, quality, and amenities.

At this stage, no exploratory analysis, feature engineering, or modeling is performed. The emphasis is on ensuring data quality and establishing a solid foundation for the following notebooks in the project.

## Table of Contents
1. [Setup & Imports](#setup)
2. [Load Data](#load-data)
3. [Initial Data Overview](#overview)
4. [Missing Value Diagnostic](#missing-diagnostic)
5. [Categorical Feature Cleaning](#cat-cleaning)
    - 5.1 ["None-Type" Categoricals](#cat-none)
    - 5.2 [Garage Categorical Features](#cat-garage)
    - 5.3 [Basement Categorical Features](#cat-basement)
    - 5.4 [True Missing Value: Electrical](#cat-electrical)
6. [Numerical Feature Cleaning](#num-cleaning)
    - 6.1 [Lot Frontage](#num-lotfrontage)
    - 6.2 [Masonry Veneer Area](#num-masvnr)
    - 6.3 [Garage Numerical Features](#num-garage)
    - 6.4 [Basement Numerical Features](#num-basement)
7. [Final Verification](#verification)
8. [Export Cleaned Dataset](#export)

## 1. Setup & Imports <a name="setup"></a>

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings("ignore")

## 2. Load Data <a name="load-data"></a>

In [2]:
df = pd.read_csv("AmesHousing.csv")
print(df.shape)
df.head(3)

(2930, 82)


,Order,PID,MS SubClass,MS Zoning,Lot Frontage,Lot Area,Street,Alley,Lot Shape,Land Contour,...,Pool Area,Pool QC,Fence,Misc Feature,Misc Val,Mo Sold,Yr Sold,Sale Type,Sale Condition,SalePrice
0,1,526301100,20,RL,141.0,31770,Pave,NaN,IR1,Lvl,...,0,NaN,NaN,NaN,0,5,2010,WD,Normal,215000
1,2,526350040,20,RH,80.0,11622,Pave,NaN,Reg,Lvl,...,0,NaN,MnPrv,NaN,0,6,2010,WD,Normal,105000
2,3,526351010,20,RL,81.0,14267,Pave,NaN,IR1,Lvl,...,0,NaN,NaN,Gar2,12500,6,2010,WD,Normal,172000


## 3. Initial Data Overview <a name="overview"></a>

In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2930 entries, 0 to 2929
Data columns (total 82 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Order            2930 non-null   int64  
 1   PID              2930 non-null   int64  
 2   MS SubClass      2930 non-null   int64  
 3   MS Zoning        2930 non-null   object 
 4   Lot Frontage     2440 non-null   float64
 5   Lot Area         2930 non-null   int64  
 6   Street           2930 non-null   object 
 7   Alley            198 non-null    object 
 8   Lot Shape        2930 non-null   object 
 9   Land Contour     2930 non-null   object 
 10  Utilities        2930 non-null   object 
 11  Lot Config       2930 non-null   object 
 12  Land Slope       2930 non-null   object 
 13  Neighborhood     2930 non-null   object 
 14  Condition 1      2930 non-null   object 
 15  Condition 2      2930 non-null   object 
 16  Bldg Type        2930 non-null   object 
 17  House Style   

## 4. Missing Value Diagnostic <a name="missing-diagnostic"></a>

In [4]:
missing = df.isnull().sum().sort_values(ascending=False)
missing = missing[missing > 0]

print(f"{len(missing)} columns contain missing values:\n")
print(missing)

27 columns contain missing values:

Pool QC           2917
Misc Feature      2824
Alley             2732
Fence             2358
Mas Vnr Type      1775
Fireplace Qu      1422
Lot Frontage       490
Garage Qual        159
Garage Yr Blt      159
Garage Cond        159
Garage Finish      159
Garage Type        157
Bsmt Exposure       83
BsmtFin Type 2      81
Bsmt Qual           80
Bsmt Cond           80
BsmtFin Type 1      80
Mas Vnr Area        23
Bsmt Full Bath       2
Bsmt Half Bath       2
Total Bsmt SF        1
BsmtFin SF 1         1
BsmtFin SF 2         1
Garage Area          1
Garage Cars          1
Bsmt Unf SF          1
Electrical           1
dtype: int64


**Reading the data dictionary.** For a lot of these columns,
`NaN` doesn't mean "unknown" — it means "this feature doesn't exist for this
property" (e.g. no alley, no garage, no basement). Those get an explicit
sentinel value. A much smaller set are genuinely missing data points that need
a real imputation strategy (e.g. `Electrical`, which is missing for exactly
one row with no indication of why). The two categories are handled separately
below.

## 5. Categorical Feature Cleaning <a name="cat-cleaning"></a>

In [5]:
cat_cols = df.select_dtypes(include=["object"]).columns

cat_cols

Index(['MS Zoning', 'Street', 'Alley', 'Lot Shape', 'Land Contour',
       'Utilities', 'Lot Config', 'Land Slope', 'Neighborhood', 'Condition 1',
       'Condition 2', 'Bldg Type', 'House Style', 'Roof Style', 'Roof Matl',
       'Exterior 1st', 'Exterior 2nd', 'Mas Vnr Type', 'Exter Qual',
       'Exter Cond', 'Foundation', 'Bsmt Qual', 'Bsmt Cond', 'Bsmt Exposure',
       'BsmtFin Type 1', 'BsmtFin Type 2', 'Heating', 'Heating QC',
       'Central Air', 'Electrical', 'Kitchen Qual', 'Functional',
       'Fireplace Qu', 'Garage Type', 'Garage Finish', 'Garage Qual',
       'Garage Cond', 'Paved Drive', 'Pool QC', 'Fence', 'Misc Feature',
       'Sale Type', 'Sale Condition'],
      dtype='object')

### 5.1 "None-Type" Categoricals <a name="cat-none"></a>

These columns use `NaN` to mean the feature is simply absent from the property, per the data dictionary. `Misc Feature` was missed in the original pass despite having 2,824 missing values — it's included here.

In [6]:
none_type_cat = {
    "Alley": "No Alley",
    "Mas Vnr Type": "No Veneer",
    "Fireplace Qu": "No Fireplace",
    "Fence": "No Fence",
    "Misc Feature": "No Feature",
    "Pool QC": "No Pool",
}

for col, fill_value in none_type_cat.items():
    df[col] = df[col].fillna(fill_value)



In [7]:
print(none_type_cat)

{'Alley': 'No Alley', 'Mas Vnr Type': 'No Veneer', 'Fireplace Qu': 'No Fireplace', 'Fence': 'No Fence', 'Misc Feature': 'No Feature', 'Pool QC': 'No Pool'}


### 5.2 Garage Categorical Features <a name="cat-garage"></a>

In [8]:
# Confirm the missing garage categoricals correspond to properties with no garage at all
df.loc[df["Garage Type"].isnull(), ["Garage Type", "Garage Cars", "Garage Area"]].head()

,Garage Type,Garage Cars,Garage Area
27,NaN,0.0,0.0
119,NaN,0.0,0.0
125,NaN,0.0,0.0
129,NaN,0.0,0.0
130,NaN,0.0,0.0


In [9]:
garage_cat = {
    "Garage Type": "No Garage",
    "Garage Finish": "No Garage",
    "Garage Qual": "No Garage",
    "Garage Cond": "No Garage",
}

for col, fill_value in garage_cat.items():
    df[col] = df[col].fillna(fill_value)

### 5.3 Basement Categorical Features <a name="cat-basement"></a>

In [10]:
# Confirm the missing basement categoricals correspond to properties with no basement
df.loc[df["Bsmt Qual"].isnull(), ["Bsmt Qual", "Total Bsmt SF", "BsmtFin SF 1"]].head()

,Bsmt Qual,Total Bsmt SF,BsmtFin SF 1
83,NaN,0.0,0.0
154,NaN,0.0,0.0
206,NaN,0.0,0.0
243,NaN,0.0,0.0
273,NaN,0.0,0.0


In [11]:
bsmt_cat = {
    "Bsmt Qual": "No Basement",
    "Bsmt Cond": "No Basement",
    "Bsmt Exposure": "No Basement",
    "BsmtFin Type 1": "No Basement",
    "BsmtFin Type 2": "No Basement",
}

for col, fill_value in bsmt_cat.items():
    df[col] = df[col].fillna(fill_value)

### 5.4 True Missing Value: Electrical <a name="cat-electrical"></a>

Unlike the columns above, a missing `Electrical` value doesn't mean "no electrical system" — every house has one. This is genuinely missing data for a single row, so it's imputed with the mode rather than a sentinel string.

In [12]:
df["Electrical"] = df["Electrical"].fillna(df["Electrical"].mode()[0])

In [13]:
# Check which categorical columns still have missing values 
# to verify whether the filling was applied correctly
cat_cols = df.select_dtypes(include=["object"]).columns.tolist()

cat_missing = df[cat_cols].isnull().sum()
cat_missing = cat_missing[cat_missing > 0].sort_values(ascending=False)

if cat_missing.empty:
    print("All categorical columns are fully filled — no missing values.")
else:
    print(f"{len(cat_missing)} categorical columns still have missing values:\n")
    print(cat_missing)
    print(f"\nTotal categorical columns: {len(cat_cols)}")

All categorical columns are fully filled — no missing values.


In [14]:
df["Pool QC"].value_counts()

Pool QC
No Pool    2917
Ex            4
Gd            4
TA            3
Fa            2
Name: count, dtype: int64

In [15]:
print(f"After filling, categorical columns have missing values: {cat_missing.sum()}")

After filling, categorical columns have missing values: 0


## 6. Numerical Feature Cleaning <a name="num-cleaning"></a>

In [16]:
Num_cols = df.select_dtypes(include=["int64", "float64"]).columns

Num_cols

Index(['Order', 'PID', 'MS SubClass', 'Lot Frontage', 'Lot Area',
       'Overall Qual', 'Overall Cond', 'Year Built', 'Year Remod/Add',
       'Mas Vnr Area', 'BsmtFin SF 1', 'BsmtFin SF 2', 'Bsmt Unf SF',
       'Total Bsmt SF', '1st Flr SF', '2nd Flr SF', 'Low Qual Fin SF',
       'Gr Liv Area', 'Bsmt Full Bath', 'Bsmt Half Bath', 'Full Bath',
       'Half Bath', 'Bedroom AbvGr', 'Kitchen AbvGr', 'TotRms AbvGrd',
       'Fireplaces', 'Garage Yr Blt', 'Garage Cars', 'Garage Area',
       'Wood Deck SF', 'Open Porch SF', 'Enclosed Porch', '3Ssn Porch',
       'Screen Porch', 'Pool Area', 'Misc Val', 'Mo Sold', 'Yr Sold',
       'SalePrice'],
      dtype='object')

### 6.1 Lot Frontage <a name="num-lotfrontage"></a>

`Lot Frontage` is the linear feet of street connected to the property. Properties in the same neighborhood tend to have similar frontage, so missing values are filled with the neighborhood median rather than the global median.

In [17]:
df["Lot Frontage"] = (
    df.groupby("Neighborhood")["Lot Frontage"]
      .transform(lambda x: x.fillna(x.median()))
)

In [18]:
# Neighborhood: Filled missing values with the mode.
# Lot Frontage: Filled remaining missing values with the overall median.
df["Neighborhood"] = df["Neighborhood"].fillna(df["Neighborhood"].mode()[0])
df["Lot Frontage"] = df["Lot Frontage"].fillna(df["Lot Frontage"].median())

In [19]:
remaining = df["Lot Frontage"].isna().sum()

print(f"Remaining missing values: {remaining}")

Remaining missing values: 0


### 6.2 Masonry Veneer Area <a name="num-masvnr"></a>

Missing `Mas Vnr Area` lines up with missing `Mas Vnr Type` — no veneer means zero area.

In [20]:
df["Mas Vnr Area"] = df["Mas Vnr Area"].fillna(0)

### 6.3 Garage Numerical Features <a name="num-garage"></a>

`Garage Cars` and `Garage Area` are straightforward zero-fills for properties with no garage.

`Garage Yr Blt` is a special case. Filling it with `0` (as in the original version of this notebook) is a latent bug: any future feature like "garage age" (`Yr Sold - Garage Yr Blt`) would silently produce an age of ~2,000 years for houses with no garage at all, instead of something sane. Filling it with the house's own `Year Built` keeps any downstream age calculation reasonable (age = 0), while `Garage Type == "NA"` still tells you the garage doesn't exist.

In [21]:
garage_num = ["Garage Cars", "Garage Area"]

for col in garage_num:
    df[col] = df[col].fillna(0)

df["Garage Yr Blt"] = df["Garage Yr Blt"].fillna(df["Year Built"])

### 6.4 Basement Numerical Features <a name="num-basement"></a>

These were **not handled in the original notebook** — `Total Bsmt SF` and `BsmtFin SF 1` were identified as still-missing for row 1341 (a property with no basement) but never actually filled, and `BsmtFin SF 2`, `Bsmt Unf SF`, `Bsmt Full Bath`, and `Bsmt Half Bath` were never checked at all despite appearing in the missing-value diagnostic above. All are zero-filled for the same reason as the categorical basement columns: no basement means zero square footage and zero basement bathrooms.

In [22]:
bsmt_num = [
    "BsmtFin SF 1",
    "BsmtFin SF 2",
    "Bsmt Unf SF",
    "Total Bsmt SF",
    "Bsmt Full Bath",
    "Bsmt Half Bath",
]

for col in bsmt_num:
    df[col] = df[col].fillna(0)

## 7. Final Verification <a name="verification"></a>

Confirms every missing value identified in Section 4 has actually been resolved before handing the dataset off.

In [23]:
remaining = df.isnull().sum()
remaining = remaining[remaining > 0]

if remaining.empty:
    print("No missing values remain.")
else:
    print("Columns still containing missing values:")
    print(remaining)

No missing values remain.


## 8. Export Cleaned Dataset <a name="export"></a>

In [24]:
df.to_csv("AmesHousing_cleaned.csv", index=False)
print("Saved cleaned dataset to AmesHousing_cleaned.csv")

Saved cleaned dataset to AmesHousing_cleaned.csv
